In [2]:
import Pkg
Pkg.add(["JuMP", "GLPK", "Format"])

using JuMP
using GLPK
using Format

   Resolving package versions...
     Project No packages added to or removed from `~/.julia/environments/v1.12/Project.toml`
    Manifest No packages added to or removed from `~/.julia/environments/v1.12/Manifest.toml`


In [4]:
function solve_min_distance_schedule(N, P; time_limit=nothing)

    # Constante BIG-M
    M = sum(P)

    model = Model(GLPK.Optimizer)

    # Limite de tempo opicional para entradas muito longas (em minutos)
    if time_limit !== nothing
        set_time_limit_sec(model, time_limit * 60.0)
    end

    #
    # Variáveis
    #

    @variable(model, D >= 0)

    # tempos de início
    @variable(model, S[1:N] >= 0)

    # variáveis binárias b(i,j), apenas para i<j
    @variable(model, b[1:N-1,2:N], Bin)

    #
    # Função objetivo
    #

    @objective(model, Min, D)

    #
    # Restrição (3)
    #

    @constraint(model, S[1] == 0)

    #
    # Restrição (1)
    #

    for i in 1:N
        @constraint(model, D >= S[i] + P[i])
    end

    #
    # Restrições (5) e (6)
    #

    for i in 1:N-1
        for j in i+1:N

            minP = min(P[i], P[j])

            # (5)
            @constraint(model,
                S[i] >= minP + S[j] - M*(1 - b[i,j])
            )

            # (6)
            @constraint(model,
                S[j] >= minP + S[i] - M*b[i,j]
            )
        end
    end

    optimize!(model)

    println()
    println("Status: ", termination_status(model))
    println()

    if termination_status(model) == MOI.OPTIMAL

        println("Makespan = ", value(D))
        println()

        println("Tempos de início")

        for i in 1:N
            println(format("Tarefa {:2d}: {:8.2f}", i, value(S[i])))
        end

        println()
        println("Valores de b(i,j)")

        for i in 1:N-1
            for j in i+1:N
                println("b($i,$j) = ", Int(round(value(b[i,j]))))
            end
        end

    else
        println("Nenhuma solução ótima encontrada.")
    end

end

# Lê os dados do arquivo upado
function read_data_from_file(filename)
    open(filename, "r") do io
        N = parse(Int, readline(io))
        P = Int[]
        for line in eachline(io)
            push!(P, parse(Int, line))
        end
        return N, P
    end
end

#
# Instância e Chamado da função
#

filename = #Insira aqui o nome do arquivo de instância upado
N, P = read_data_from_file(filename)

#
# Tamanho da instância, instância, limite de tempo (opicional)
#
solve_min_distance_schedule(N, P, time_limit=3.0)



Status: TIME_LIMIT

Nenhuma solução ótima encontrada.
